# 第 4 章 · 深度学习基础：把正文的手算搬进代码

> 对应正文：[神经网络是怎么炼成的](../docs/3-深度学习/04-深度学习基础/01-神经网络是怎么炼成的.md) · [训练一个模型的完整流程](../docs/3-深度学习/04-深度学习基础/02-训练一个模型的完整流程.md) · [返回本章 README](../docs/3-深度学习/04-深度学习基础/README.md) · [返回 notebooks 总览](./README.md)

本 notebook 把正文两页的"动手试试"与"数学深潜"扩展成 7 个可复现实验：手写前向/反向传播并与正文数字逐位对照、激活函数饱和现场、ReLU 网络分段线性折线、三种优化器下降轨迹、PyTorch 训练循环四步模板、学习率三档命运。

**环境说明**

- 实验 1~5、7 只依赖 `numpy` + `matplotlib`，离线可跑：数据全部由代码生成、随机种子固定，从上到下执行一遍即复现全部结果；
- 实验 6 需要本地 PyTorch 环境，代码已按标准写法给出并注明本地运行预期；未安装 torch 时该单元自动打印跳过提示，不影响其余单元执行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 固定随机种子，保证每次运行结果一致
np.random.seed(42)

# matplotlib 中文显示设置（Windows 用 SimHei/微软雅黑；Mac 可换 Arial Unicode MS）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False    # 让负号正常显示
# 对数轴刻度走 mathtext：把数学字体钉回 DejaVu Sans，避免中文字体缺负号字形（U+2212）
plt.rcParams["mathtext.fontset"] = "dejavusans"
for _k in ("mathtext.rm", "mathtext.it", "mathtext.bf", "mathtext.sf", "mathtext.tt", "mathtext.cal"):
    plt.rcParams[_k] = "DejaVu Sans"

print("numpy 版本:", np.__version__, "；matplotlib 中文字体已设置")

## 实验 1：手写前向传播——迷你网络 2→2→1 逐层算

**目标**：用 numpy 复刻正文《神经网络是怎么炼成的》的算例——输入 (1.0, 2.0)，2 个隐藏神经元 + 1 个输出神经元（全部 sigmoid）。前向传播只有三步：加权求和 → 过激活 → 喂给下一层。跑完与正文的 0.818 / 0.786 / 0.655 逐位对照。

In [ ]:
# ========= 实验 1：手写前向传播（与正文算例逐位对照） =========
def sigmoid(z):
    """sigmoid 激活：把任意实数压到 (0, 1)"""
    return 1.0 / (1.0 + np.exp(-z))

# 网络结构 2 → 2 → 1，参数与正文完全一致
x = np.array([1.0, 2.0])                     # 输入
W1 = np.array([[0.6, 0.4],                   # 隐藏神经元 h1 的权重 (w11, w12)
               [-0.2, 0.8]])                 # 隐藏神经元 h2 的权重 (w21, w22)
b1 = np.array([0.1, -0.1])                   # 隐藏层偏置
W2 = np.array([1.5, -1.0])                   # 输出神经元权重 (v1, v2)
b2 = 0.2                                     # 输出偏置

# 第 ① 步：隐藏层加权求和（一行矩阵乘法 = 正文手算的两条 z 公式）
z_h = W1 @ x + b1
print("隐藏层加权和 z_h =", np.round(z_h, 4), "   ← 正文手算: [1.5, 1.3]")

# 第 ② 步：sigmoid 激活
h = sigmoid(z_h)
print("隐藏层激活    h  =", np.round(h, 4), "  ← 正文手算: [0.818, 0.786]")

# 第 ③ 步：输出层加权求和 + 激活
z_o = W2 @ h + b2
y = sigmoid(z_o)
print("输出层加权和 z_o =", round(float(z_o), 4), "    ← 正文手算: 0.641")
print("最终输出      y  =", round(float(y), 4), "    ← 正文手算: 0.6549")

assert round(float(y), 4) == 0.6549
print("\n与正文 0.6549 完全一致 ✓  这个数字将在实验 2 里继续被『追责』")

**小结**：一层 = 一次矩阵乘法 + 一次逐元素激活，`W1 @ x + b1` 一行就是正文手算的两条 z 公式。输出 0.6549 与正文完全一致——下一节就让这个数字"被追责"。

## 实验 2：手写反向传播——梯度怎么"追责"回每个权重

**目标**：目标值 t = 0.95，误差约 0.3。按正文 δ 递推公式（δ₂ = (y−t)·y(1−y)；δ₁ = (W₂ᵀδ₂)⊙h(1−h)）手写全部 9 个参数的梯度，与正文的 −0.0149 等数字对照；再用数值梯度（中心差分）验证手推无误，误差应远小于 1e-6。

In [ ]:
# ========= 实验 2：手写反向传播 + 数值梯度验证 =========
t = 0.95                                     # 目标值：误差 y - t ≈ -0.3

def forward_all(W1_, b1_, W2_, b2_):
    """前向：返回 (输出 y, 隐藏层激活 h)"""
    h_ = sigmoid(W1_ @ x + b1_)
    return sigmoid(W2_ @ h_ + b2_), h_

def loss_value(W1_, b1_, W2_, b2_):
    """平方损失 L = 1/2 (y - t)^2"""
    y_, _ = forward_all(W1_, b1_, W2_, b2_)
    return 0.5 * (y_ - t) ** 2

# ---------- 解析梯度：δ 递推（正文的"追责链"） ----------
y_hat, h_act = forward_all(W1, b1, W2, b2)
print("五环责任因子（与正文表格逐项对照）：")
print(f"  ① 输出误差 y - t    = {y_hat - t:+.4f}  （正文 -0.295）")
print(f"  ② 输出灵敏度 y(1-y) = {y_hat * (1 - y_hat):.4f}   （正文 0.226）")
print(f"  ③ 权重传递 v1       = {W2[0]:.4f}   （正文 1.5）")
print(f"  ④ h1 灵敏度 h(1-h)  = {h_act[0] * (1 - h_act[0]):.4f}   （正文 0.149）")
print(f"  ⑤ 输入 x1           = {x[0]:.4f}   （正文 1.0）")

delta2 = (y_hat - t) * y_hat * (1 - y_hat)          # 输出层误差信号 δ₂
grad_W2 = delta2 * h_act                            # ∂L/∂v = δ₂ × 隐藏层激活
grad_b2 = np.array([delta2])                        # ∂L/∂b₂ = δ₂ 本身
delta1 = (W2 * delta2) * h_act * (1 - h_act)        # δ₁ = (W₂ᵀδ₂) ⊙ h(1-h)，一维时转置=逐元素乘
grad_W1 = np.outer(delta1, x)                       # ∂L/∂W₁ = δ₁ × 该层输入（外积）
grad_b1 = delta1

print(f"\n五环相乘 ∂L/∂w11 = {grad_W1[0, 0]:+.4f}  （正文 -0.0149）")
print("全部 9 个梯度（与正文完整算例对照；末位 ±0.0001 的出入来自正文的舍入递推）：")
print(f"  w11 = {grad_W1[0, 0]:+.4f}（正文 -0.0149）  w12 = {grad_W1[0, 1]:+.4f}（正文 -0.0298）")
print(f"  w21 = {grad_W1[1, 0]:+.4f}（正文 +0.0112）  w22 = {grad_W1[1, 1]:+.4f}（正文 +0.0225）")
print(f"  b_h1 = {grad_b1[0]:+.4f}（正文 -0.0149）  b_h2 = {grad_b1[1]:+.4f}（正文 +0.0112）")
print(f"  v1 = {grad_W2[0]:+.4f}（正文 -0.0545）   v2 = {grad_W2[1]:+.4f}（正文 -0.0524）")
print(f"  b_o = {grad_b2[0]:+.4f}（正文 -0.0667）")

# ---------- 学习率 0.5 迈一步（正文更新算例） ----------
lr = 0.5
print(f"\n一步更新：w11 ← 0.6 - 0.5 × ({grad_W1[0, 0]:+.4f}) = {0.6 - lr * grad_W1[0, 0]:.4f}（正文 ≈ 0.607）")

# ---------- 数值梯度验证：中心差分 (L(θ+ε) - L(θ-ε)) / 2ε ----------
eps = 1e-5
analytic = {"W1": grad_W1, "b1": grad_b1, "W2": grad_W2, "b2": grad_b2}
base = {"W1": W1.copy(), "b1": b1.copy(), "W2": W2.copy(), "b2": np.array([b2])}
max_err = 0.0
for key in base:
    arr = base[key]
    num = np.zeros_like(arr, dtype=float)
    it = np.nditer(arr, flags=["multi_index"])
    for _ in it:                                   # 逐个参数扰动一次
        idx = it.multi_index
        orig = arr[idx]
        arr[idx] = orig + eps
        lp = loss_value(base["W1"], base["b1"], base["W2"], base["b2"])
        arr[idx] = orig - eps
        lm = loss_value(base["W1"], base["b1"], base["W2"], base["b2"])
        arr[idx] = orig
        num[idx] = (lp - lm) / (2 * eps)
    err = np.max(np.abs(num - analytic[key]))
    max_err = max(max_err, err)
    print(f"参数 {key:>2}：数值梯度 vs 解析梯度 最大误差 = {err:.2e}")
print(f"\n全部 9 个参数的最大误差 = {max_err:.2e}")
assert max_err < 1e-6
print("误差 < 1e-6 ✓ 手推的链式法则与数值求导完全一致")

**小结**：反向传播 = 链式法则的工业化——定义误差信号 δ 逐层递推，每层参数梯度都是"δ × 该层输入"。正文三个数字全部对上：∂L/∂w11 = −0.0149、一步更新 w11 → 0.607、数值梯度验证误差 < 1e-6。还能看到 w12 的梯度约为 w11 的 2 倍（梯度正比于这条线上流过的信号量），h2 通道梯度为正（误差经 v2 = −1.0 回传时变号）——符号与大小都有物理含义，不是随机数。

## 实验 3：激活函数梯度现场——sigmoid 的"饱和"证据

**目标**：正文说 sigmoid 导数峰值仅 0.25、|z| 很大时"失聪"。这里打印 σ′(z) 在不同 z 处的现场数值，算一次 10 层连乘（梯度消失的乘法现场），并画出三种激活函数及其导数曲线——选激活函数看的正是导数这张"梯度传导率"图。

In [ ]:
# ========= 实验 3：激活函数梯度现场 =========
def sigmoid_grad(z):
    """σ'(z) = σ(z)(1 - σ(z))"""
    s = sigmoid(z)
    return s * (1 - s)

print("sigmoid 导数现场（= 梯度传导率）：")
for z in [0, 2, 5, 10, -5, -10]:
    print(f"  z = {z:>3} → σ'(z) = {sigmoid_grad(z):.3e}")
print(f"\nσ'(±10) ≈ 4.5e-5 已接近失聪；若 10 层都在 z ≈ 10，梯度连乘：")
print(f"  (4.5e-5)^10 ≈ {sigmoid_grad(10) ** 10:.2e} —— 梯度数学上存在、数值上等于 0")
print("对照：ReLU 在 z > 0 处导数恒为 1，连乘 10 层仍是 1（深网络隐藏层默认 ReLU 的原因）")

z = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z, sigmoid(z), label="Sigmoid")
axes[0].plot(z, np.tanh(z), label="Tanh")
axes[0].plot(z, np.maximum(z, 0), label="ReLU")
axes[0].set(title="三种激活函数曲线", xlabel="z（加权和）", ylabel="激活值")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(z, sigmoid_grad(z), label="Sigmoid 的导数（峰值仅 0.25）")
axes[1].plot(z, 1 - np.tanh(z) ** 2, label="Tanh 的导数（峰值 1，两端仍饱和）")
axes[1].step(z, (z > 0).astype(float), where="mid", label="ReLU 的导数（正区间恒 1）")
axes[1].set(title="导数曲线 = 梯度传导率", xlabel="z", ylabel="导数")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**小结**：sigmoid 峰值传导率仅 0.25，|z| > 5 就掉到 1e-3 以下，10 层连乘直接到 1e-43 量级——"梯度消失"不是比喻，是连乘的算术。tanh 零中心、峰值 1，深了照样饱和；ReLU 正区间传导率恒 1，是深网络的默认起点。

## 实验 4：ReLU 网络是分段线性——正文"数学深潜"的可跑版

**目标**：一维输入、3 个隐藏单元的网络 g(x) = max(0, x+1) + max(0, −x+1) + 2·max(0, x)。正文手算出 4 段折线（斜率 −1 / 0 / +2 / +3）与抽查点 g(2)=7、g(−2)=3。这里逐段实测斜率、画折线图逐项验证。

In [ ]:
# ========= 实验 4：ReLU 网络的分段线性 =========
def g_relu(x):
    """g(x) = max(0, x+1) + max(0, -x+1) + 2·max(0, x)：3 个隐藏单元的加权求和"""
    return np.maximum(0, x + 1) + np.maximum(0, -x + 1) + 2.0 * np.maximum(0, x)

# 正文抽查：每段内部"一个公式管到底"
print("g(2)  =", g_relu(2.0), "  ← 段④公式 3x+1 = 7")
print("g(0)  =", g_relu(0.0), "  ← 折点连续：段②给 2，段③给 2+2×0 = 2")
print("g(-2) =", g_relu(-2.0), " ← 段①公式 1-x = 3")

# 逐段实测斜率（在区间内部取一小段算 Δg/Δx），对照正文表格
print("\n各段斜率（实测 vs 正文表格）：")
segments = [(-3, -1), (-1, 0), (0, 1), (1, 3)]
expect_slopes = [-1, 0, 2, 3]
for (lo, hi), s_exp in zip(segments, expect_slopes):
    a, b = lo + 1e-4, hi - 1e-4                   # 缩进区间内部，避开折点
    s_meas = (g_relu(b) - g_relu(a)) / (b - a)
    print(f"  区间 ({lo:+d}, {hi:+d})：实测斜率 {s_meas:+.2f}，正文 {s_exp:+d}")

xs = np.linspace(-3, 3, 601)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, g_relu(xs), lw=2.5, color="steelblue")
for kp in [-1, 0, 1]:                             # 三个折点：隐藏单元开启/关闭的位置
    ax.axvline(kp, color="gray", ls="--", alpha=0.6)
pts_x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
ax.scatter(pts_x, g_relu(pts_x), color="crimson", zorder=3)
ax.annotate("段① 斜率 -1", (-2.95, 3.7), fontsize=10)
ax.annotate("段② 斜率 0", (-0.92, 2.12), fontsize=10)
ax.annotate("段③ 斜率 +2", (0.08, 3.3), fontsize=10)
ax.annotate("段④ 斜率 +3", (1.45, 5.7), fontsize=10)
ax.set(title="ReLU 网络的 4 段折线：g(x)=max(0,x+1)+max(0,-x+1)+2·max(0,x)",
       xlabel="输入 x（虚线 = 折点 -1、0、1）", ylabel="g(x)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**小结**：三个折点 −1、0、1 把数轴切成 4 段，每段内 g(x) 严格是一条直线，实测斜率 −1 / 0 / +2 / +3 与正文表格逐段一致；跨过折点斜率恰好跳 u_j·w_j（跨过 0 点 h₃ 开启，斜率 0 → 2 跳了 u₃w₃ = 2）。ReLU 网络的全部参数就花在"折痕摆哪 + 每段斜率多少"两件事上——正文"切空间机器"的几何图景可以跑、可以量了。

## 实验 5：优化器对比——SGD / Momentum / Adam 同跑一座"狭长山沟"

**目标**：正文"数学深潜"用过的二次碗 f(w) = ½(w₁² + 100·w₂²)（条件数 κ = 100：平方向 λ=1 慢爬，陡方向 λ=100 锁死步长上限 2/100）。用 numpy 手写三种优化器各跑 120 步：SGD 取 η=0.02 时陡方向公比 |1−0.02×100| = 1，永远在两岸横跳（正文表格的现场）；Momentum 用谱半径 √β ≈ 0.949 匀速收缩；Adam 逐坐标自适应步长。交互版：[playground 梯度下降炼丹炉](../playground/gradient-descent.html)。

In [ ]:
# ========= 实验 5：优化器对比（numpy 手写三种优化器） =========
def f_bowl(w):
    """狭长山沟：平方向 λ=1，陡方向 λ=100，条件数 κ=100"""
    return 0.5 * (w[0] ** 2 + 100.0 * w[1] ** 2)

def grad_bowl(w):
    return np.array([w[0], 100.0 * w[1]])

def run_sgd(w0, lr, steps):
    w, path = w0.copy(), [w0.copy()]
    for _ in range(steps):
        w = w - lr * grad_bowl(w)                    # 盲人下山：每步只看脚下
        path.append(w.copy())
    return np.array(path)

def run_momentum(w0, lr, beta, steps):
    w, v, path = w0.copy(), np.zeros_like(w0), [w0.copy()]
    for _ in range(steps):
        v = beta * v + grad_bowl(w)                  # 速度累积惯性（正文公式）
        w = w - lr * v
        path.append(w.copy())
    return np.array(path)

def run_adam(w0, lr, steps, beta1=0.9, beta2=0.999):
    w, m, v, path = w0.copy(), np.zeros_like(w0), np.zeros_like(w0), [w0.copy()]
    for t in range(1, steps + 1):
        g = grad_bowl(w)
        m = beta1 * m + (1 - beta1) * g              # 一阶矩：梯度的均值
        v = beta2 * v + (1 - beta2) * g ** 2         # 二阶矩：梯度的波动幅度
        m_hat = m / (1 - beta1 ** t)                 # 偏差修正
        v_hat = v / (1 - beta2 ** t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + 1e-8) # 每个坐标自适应步长
        path.append(w.copy())
    return np.array(path)

w0 = np.array([8.0, 1.5])                            # 从山腰出发
steps = 120
paths = {
    "SGD (η=0.02)": run_sgd(w0, 0.02, steps),        # 陡方向公比恰为 1：永不收敛
    "Momentum (η=0.02, β=0.9)": run_momentum(w0, 0.02, 0.9, steps),
    "Adam (η=0.1)": run_adam(w0, 0.1, steps),
}

print("120 步后的战况：")
for name, p in paths.items():
    dist = np.linalg.norm(p[-1])
    print(f"  {name:<26} 距谷底 ‖w‖ = {dist:6.3f}，损失 = {f_bowl(p[-1]):.3e}")
print("→ SGD 在陡方向永远 ±1.5 横跳（w₂ 公比 |1-0.02×100| = 1，正文表格现场）；")
print("  Momentum 谱半径 √0.9 ≈ 0.949，两个方向匀速收缩；")
print("  Adam 不需要知道 λ，逐坐标配步长，最后落进 η 量级的『噪声球』")

# 左图：等高线 + 三条轨迹；右图：损失曲线（对数轴）
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
gx = np.linspace(-9, 9, 200)
gy = np.linspace(-2.5, 2.5, 200)
GX, GY = np.meshgrid(gx, gy)
axes[0].contour(GX, GY, 0.5 * (GX ** 2 + 100 * GY ** 2), levels=30, cmap="Blues", alpha=0.6)
colors = ["tab:red", "tab:green", "tab:purple"]
for (name, p), c in zip(paths.items(), colors):
    axes[0].plot(p[:, 0], p[:, 1], color=c, lw=1.6, label=name)
axes[0].scatter([0], [0], marker="*", s=150, color="gold", edgecolor="k", zorder=3, label="谷底")
axes[0].set(title="三条下降轨迹（等高线越密的方向越陡）",
            xlabel="w1（平方向 λ=1）", ylabel="w2（陡方向 λ=100）")
axes[0].legend(fontsize=9)
for (name, p), c in zip(paths.items(), colors):
    axes[1].semilogy(np.arange(len(p)), [f_bowl(w) for w in p], color=c, label=name)
from matplotlib.ticker import FuncFormatter
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))   # 纯文本刻度
axes[1].set(title="损失曲线（对数轴）：谁先下到谷底", xlabel="步数", ylabel="损失 f(w)")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**小结**：同一座 κ=100 的山沟——SGD 在 η=0.02 处陡方向公比恰为 1，120 步后损失仍 ~113（永不收敛的 Z 字）；Momentum 靠惯性把条件数"开方"，一路收缩到 1e-4 量级；Adam 不需要任何 λ 的先验，自己配平各方向步长，快速落到噪声球后停住。这正是正文的结论：动量解"狭长山沟"，Adam 赢在开箱即用；想在真实任务里精调出更好终值，还是 SGD 系 + 学习率调度。动手拖参数的版本见 [playground/gradient-descent.html](../playground/gradient-descent.html)。

## 实验 6：训练循环四步模板（需本地 PyTorch 环境）

**目标**：正文《训练一个模型的完整流程》的五行注释——前向、打分、清梯度、反向、更新——落成四步复用模板（造数据 → 模型+损失+优化器 → 循环 → 评估），用它训一个 MNIST 上的 MLP（784 → 256 → 10）。

> **需本地 PyTorch 环境，代码已按标准写法给出**；未安装 torch 的环境运行本单元会打印跳过提示，不影响其余单元。**本地运行预期**：2 个 epoch 后 MNIST 测试集准确率 98%+（CPU 数分钟）；真实 MNIST 首次运行需联网下载（约 11 MB）。

In [ ]:
# ========= 实验 6（需本地 PyTorch 环境）：训练循环四步模板 =========
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    HAS_TORCH = True
except Exception:                # 未安装或安装损坏（如 DLL 加载失败）都优雅跳过
    HAS_TORCH = False
    print("未检测到可用的 PyTorch → 跳过实跑。本地 pip install torch torchvision 后重跑本单元即可。")

if HAS_TORCH:
    torch.manual_seed(0)                              # 固定种子，结果可复现

    # ---- 第 0 步：造数据（真实 MNIST，首跑自动下载约 11MB、需联网） ----
    transform = transforms.Compose([
        transforms.ToTensor(),                       # 像素缩放到 [0, 1]
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 的均值/标准差
    ])
    train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
    test_set = datasets.MNIST("./data", train=False, transform=transform)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=256)

    # ---- 第 1 步：模型 + 损失 + 优化器（MLP：784 → 256 → 10） ----
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 10),
    )
    loss_fn = nn.CrossEntropyLoss()                   # 多分类用交叉熵
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)   # 默认首选 Adam

    # ---- 第 2 步：训练循环（前向 → 打分 → 清梯度 → 反向 → 更新） ----
    for epoch in range(2):
        model.train()
        for X, y in train_loader:
            loss = loss_fn(model(X), y)   # ① 前向传播 + 损失打分
            optimizer.zero_grad()         # ② 清空上一轮梯度（新手最常忘！）
            loss.backward()               # ③ 反向传播：算出所有梯度
            optimizer.step()              # ④ 优化器：更新所有权重
        print(f"epoch {epoch + 1}  最后一批 loss = {loss.item():.4f}")

    # ---- 第 3 步：评估 ----
    model.eval()
    correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            correct += (model(X).argmax(1) == y).sum().item()
    print(f"MNIST 测试集准确率 = {correct / len(test_set):.2%}（本地预期 98%+）")

**小结**：四步模板（造数据 → 模型+损失+优化器 → 循环 → 评估）从本章一路复用到视觉、时序、NLP 各章——只是换了模型结构和数据，骨架不变；`zero_grad → backward → step` 三连是新手最常漏的顺序。MNIST 对 MLP 是简单任务，2 个 epoch 到 98%+ 是健康基线；达不到就按正文排查口诀"数据 → 损失 → 学习率 → 梯度 → 容量"逐项体检。

## 实验 7：学习率三档实验——过小 / 合适 / 过大的三种命运

**目标**：正文"小实验"的复现：同一份回归数据、同一个模型（numpy 手写线性回归 + 全批量梯度下降），只改学习率。特征故意不做归一化（均值 ~8、幅度大 → 损失面最陡方向曲率 λmax ≈ 65，学习率生死线 2/λmax ≈ 0.031，正文数学深潜的公式）。三档：η=1e-4 慢如蜗牛、η=2e-2 匹配坡度、η=3.5e-2 刚越过生死线——来回震荡、越震越远。

In [ ]:
# ========= 实验 7：学习率三档实验（numpy 线性回归 + 全批量梯度下降） =========
rng = np.random.default_rng(0)
n_per_class = 200
# 特征故意不归一化（均值 ~8、幅度大 → 曲率大、学习率安全上限低）
X_pos = rng.normal([8.0, 8.0], 1.2, (n_per_class, 2))
X_neg = rng.normal([0.0, 0.0], 1.2, (n_per_class, 2))
X_lr = np.hstack([np.ones((2 * n_per_class, 1)), np.vstack([X_pos, X_neg])])   # 加偏置列
w_true = np.array([1.0, 3.0, -2.0])
y_lr = X_lr @ w_true + rng.normal(0, 0.5, 2 * n_per_class)                     # 观测 = 真值 + 噪声

def mse_loss(w):
    """平均平方损失 L = 1/2n · Σ(预测 - y)²（回归损失不饱和——大学习率会诚实地发散）"""
    return 0.5 * np.mean((X_lr @ w - y_lr) ** 2)

def mse_grad(w):
    return X_lr.T @ (X_lr @ w - y_lr) / len(y_lr)

lam_max = np.linalg.eigvalsh(X_lr.T @ X_lr / len(y_lr))[-1]
print(f"损失面最陡方向的曲率 λmax = {lam_max:.1f} → 学习率生死线 2/λmax = {2 / lam_max:.4f}（正文数学深潜）")
print(f"噪声地板 ≈ 0.5×噪声方差 = {0.5 * 0.25:.3f}：损失降到这里就算到顶了\n")

epochs = 300
lr_settings = [("过小 η=1e-4", 1e-4), ("合适 η=2e-2", 2e-2), ("过大 η=3.5e-2", 3.5e-2)]
curves = {}
for name, lr_val in lr_settings:
    w = np.zeros(X_lr.shape[1])
    losses = []
    for k in range(epochs):
        v = mse_loss(w)
        if not np.isfinite(v) or v > 1e12:          # 越过 1e12 视为发散，停止记录
            print(f"{name}：第 {k} 步损失越过 1e12 —— 训练发散（来回震荡、越震越远）")
            break
        losses.append(v)
        w = w - lr_val * mse_grad(w)                # 三种命运的区别只在 η 这一个数
    curves[name] = np.array(losses)
    j = min(10, len(losses) - 1)
    print(f"{name}：首 epoch loss = {losses[0]:.2f}，第 10 = {losses[j]:.3f}，"
          f"最后 = {losses[-1]:.4f}")

from matplotlib.ticker import FuncFormatter
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for name, curve in curves.items():
    ax.semilogy(np.arange(len(curve)), curve, lw=2, label=name)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))   # 纯文本刻度
ax.set(title="学习率三档：同一起点、同一损失面、三种命运", xlabel="epoch", ylabel="平方损失（对数轴）")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**小结**：η=1e-4 走了 300 个 epoch 还卡在半山腰（27.5 → 8.6）；η=2e-2 平滑收敛到 0.107，正好贴住噪声地板 0.125——模型已到上限；η=3.5e-2 只比生死线 2/λmax ≈ 0.031 大一点点，每步误差放大 |1−ηλ| ≈ 1.28 倍，震荡着冲上 1e12。正文数学深潜的 |1−ηλ|<1 在这里看得见摸得着——调参从学习率开始，这是每个炼丹师的第一课。

## 改参数建议（拿这个 notebook 当实验台）

1. **换权重再追责**：把实验 1/2 的 W1、W2 改成别的数字（如 `W2[0] = -1.5`），重跑看 9 个梯度的符号怎么翻转——"误差经负权重回传时变号"会自己跳出来；数值梯度验证那行 `assert` 会继续替你把关。
2. **加深饱和现场**：把实验 3 的连乘层数从 10 改成 20、30，或把 z 从 10 改成 5、20，看梯度从 1e-43 一路掉到接近机器零——"渐变的饱和过程"亲眼看一遍。
3. **给折纸机加单元**：在实验 4 的 `g_relu` 里再加一项 `0.5 * np.maximum(0, 2*x - 2)`，数一数折点从 3 个变 4 个、段数变 5 段——验证"K 个单元最多折出 K+1 段"。
4. **优化器调参**：实验 5 把 SGD 的 η 从 0.02 改成 0.021（越过 2/λ_max = 0.02），亲眼看一步发散；再把 β 改 0.99、Adam 的 η 改大 10 倍，比较谁对超参更敏感。
5. **本地换结构再套模板**：实验 6 本地把隐藏层 256 改成 64/1024 或叠成两层，对比 2 个 epoch 的准确率与训练时间；再用 `nn.Sigmoid()` 换掉 ReLU 重训，观察收敛明显变慢（实验 3 的饱和在现场发作）。